# Phase 2 — run the scripts on Kaggle
Runs the three scripts of the repository in order and uploads both checkpoints to the Hugging Face Hub.

**Before *Run All*:** *Settings → Accelerator: GPU T4*, *Internet: On*; *Add Input* → the project dataset
(the repository folder); *Add-ons → Secrets* → `HF_TOKEN` (a Hugging Face **write** token), switched on for this notebook.
Full run ≈ 40 min.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
src = sorted(Path("/kaggle/input").glob("**/config.yaml"))[0].parent      # the uploaded repository
PROJECT = Path("/kaggle/working/abcd")
if not PROJECT.exists():
    shutil.copytree(src, PROJECT)
os.chdir(PROJECT)
print("project:", PROJECT)
!nvidia-smi --query-gpu=name --format=csv,noheader
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN found:", bool(os.environ["HF_TOKEN"]))

## 1. Continued pretraining (`train_cpt.py`, ≈ 30 min)

In [ ]:
!python train_cpt.py --push

## 2. SFT (`train_sft.py`, ≈ 2 min)

In [ ]:
!python train_sft.py --push

## 3. Score the three models on the 30 test questions (`score.py`, ≈ 5 min)

In [ ]:
!python score.py
print(open('SCOREBOARD.md').read())

## 4. For the demo video
The same two test questions answered by the base, continued-pretrained and SFT models, then `score.py` with its results.

In [ ]:
!python score.py --ask SP-07 SP-25

In [ ]:
!python score.py --no-ppl

## 5. Download the new results

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/scripts_run_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in [Path("SCOREBOARD.md"), *Path("results").rglob("*"), *Path("figures").glob("*")]:
        if p.is_file():
            z.write(p)
print("download: /kaggle/working/scripts_run_results.zip")